# 🚀 SpaceGen AI - Free Cloud GPU 3D Reconstruction Backend
Run high-speed 3D Gaussian Splatting and COLMAP on a **Free Google Colab Tesla T4 GPU**.

### 📌 Instructions:
1. In the Colab menu, click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Save.
2. Run **Cell 1** to install dependencies (~40s).
3. Run **Cell 2** to start the backend and copy your **public URL**.

In [ ]:
# CELL 1: Install COLMAP, Nerfstudio, and dependencies
import os, subprocess, sys
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

!apt-get update -qq
!apt-get install -y -qq colmap ffmpeg libgl1-mesa-glx

# Stub open3d for Python 3.13 compatibility
!python3 -c "import site, os; p = site.getsitepackages()[0] + '/open3d'; os.makedirs(p, exist_ok=True); open(p + '/__init__.py', 'w').write('def __getattr__(name):\n    return object\n')"

# Completely purge ALL opencv variants first, then install headless
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless 2>/dev/null
# Remove any leftover cv2 dirs from site-packages
!python3 -c "import site, shutil, os; [shutil.rmtree(os.path.join(d, 'cv2'), ignore_errors=True) for d in site.getsitepackages()]"

!pip install -q ninja fastapi uvicorn python-multipart gsplat tyro jaxtyping mediapy msgpack msgpack_numpy rich rawpy scipy av h5py plyfile appdirs scikit-image
!pip install --force-reinstall --no-cache-dir opencv-python-headless
!pip install -q nerfstudio --no-deps

# Verify opencv works
!python3 -c "import cv2; print(f'✅ OpenCV {cv2.__version__} — imread: {hasattr(cv2, \"imread\")}')"

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print('✅ Environment ready! Now run Cell 2.')

In [ ]:
# CELL 2: Start Cloud GPU FastAPI Backend with Public Tunnel
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

import uuid
import shutil
import subprocess
import threading
import time
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI(title="SpaceGen Cloud GPU Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DATA_ROOT = Path("/content/spacegen_data/reconstruction")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
JOBS = {}

def run_cmd(cmd):
    print(f"[RUNNING] {' '.join(cmd)}")
    env = os.environ.copy()
    env['QT_QPA_PLATFORM'] = 'offscreen'
    env['DISPLAY'] = ''
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        line = line.strip()
        if line:
            print(line)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}")

@app.get("/")
async def root():
    return {"status": "online", "device": "Tesla T4 GPU", "service": "SpaceGen Cloud Backend"}

def _worker(job_id: str, video_path: Path, job_dir: Path):
    try:
        processed = job_dir / "processed"
        output = job_dir / "output"
        export_dir = job_dir / "export"
        
        JOBS[job_id]["step"] = "Estimating camera poses with COLMAP"
        JOBS[job_id]["progress"] = 30
        run_cmd(["ns-process-data", "video", "--data", str(video_path), "--output-dir", str(processed), "--num-frames-target", "100"])
        
        JOBS[job_id]["step"] = "Training Gaussian Splatting on Tesla T4 GPU (15,000 steps)"
        JOBS[job_id]["progress"] = 65
        run_cmd([
            "ns-train", "splatfacto",
            "--data", str(processed),
            "--output-dir", str(output),
            "--max-num-iterations", "15000",
            "--pipeline.model.num-downscales", "1",
            "--viewer.quit-on-train-completion", "True"
        ])
        
        JOBS[job_id]["step"] = "Exporting 3D Gaussian Splat PLY"
        JOBS[job_id]["progress"] = 90
        configs = sorted(output.glob("**/config.yml"), key=lambda p: p.stat().st_mtime, reverse=True)
        if configs:
            run_cmd(["ns-export", "gaussian-splat", "--load-config", str(configs[0]), "--output-dir", str(export_dir)])
        
        JOBS[job_id]["status"] = "completed"
        JOBS[job_id]["progress"] = 100
        JOBS[job_id]["step"] = "Scene ready"
        JOBS[job_id]["log"] = "Gaussian Splatting scene completed on Tesla T4 GPU."
        JOBS[job_id]["scene_path"] = str(export_dir)
        print(f"\n🎉 JOB {job_id} COMPLETED SUCCESSFULLY!")
    except Exception as e:
        print(f"\n❌ JOB {job_id} FAILED: {e}")
        JOBS[job_id]["status"] = "failed"
        JOBS[job_id]["log"] = str(e)

@app.post("/api/reconstruction/jobs", status_code=202)
async def create_job(video: UploadFile = File(...)):
    job_id = uuid.uuid4().hex[:12]
    job_dir = DATA_ROOT / job_id
    job_dir.mkdir(parents=True, exist_ok=True)
    video_path = job_dir / "capture.mp4"
    with video_path.open("wb") as f:
        shutil.copyfileobj(video.file, f)
    
    JOBS[job_id] = {
        "job_id": job_id,
        "status": "running",
        "progress": 10,
        "step": "Processing video keyframes",
        "splat_url": f"/api/reconstruction/jobs/{job_id}/model"
    }
    threading.Thread(target=_worker, args=(job_id, video_path, job_dir), daemon=True).start()
    return JOBS[job_id]

@app.get("/api/reconstruction/jobs/{job_id}")
async def get_job(job_id: str):
    if job_id in JOBS:
        return JOBS[job_id]
    ply = DATA_ROOT / job_id / "export" / "splat.ply"
    if ply.exists():
        return {
            "job_id": job_id,
            "status": "completed",
            "progress": 100,
            "step": "Scene ready",
            "splat_url": f"/api/reconstruction/jobs/{job_id}/model"
        }
    raise HTTPException(404, "Job not found")

@app.get("/api/reconstruction/jobs/latest")
async def get_latest_job():
    if JOBS:
        return list(JOBS.values())[-1]
    job_dirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True)
    for d in job_dirs:
        if (d / "export" / "splat.ply").exists():
            return {
                "job_id": d.name,
                "status": "completed",
                "progress": 100,
                "step": "Scene ready",
                "splat_url": f"/api/reconstruction/jobs/{d.name}/model"
            }
    raise HTTPException(404, "No jobs yet")

@app.get("/api/reconstruction/jobs/{job_id}/model")
async def get_model(job_id: str):
    ply = DATA_ROOT / job_id / "export" / "splat.ply"
    if ply.exists():
        return FileResponse(path=ply, media_type="application/octet-stream", filename="splat.ply", headers={"Access-Control-Allow-Origin": "*"})
    raise HTTPException(404, "Model file not found")

# Start Cloudflare Tunnel in background
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

cloud_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        for part in line.split():
            if "https://" in part and "trycloudflare.com" in part:
                cloud_url = part.strip()
                break
        if cloud_url:
            break
    time.sleep(0.5)

print("\n=======================================================")
print(f"🚀 YOUR FREE CLOUD BACKEND URL:")
print(f"   {cloud_url or 'Starting...'}")
print(f"👉 Copy this URL and paste it in your local frontend/.env.local:")
print(f"   NEXT_PUBLIC_API_URL={cloud_url}")
print("=======================================================\n")

# Run uvicorn on Colab's active event loop
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()
